# Week 11 — LLM Systems

**Focus:** How do context, adaptation, and inference budgets change an LLM system?

*Live-coding notebook*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/smc77/uc_finmlai/blob/main/lectures/week11/week11_demos.ipynb)

## What this notebook is about

A language-model application can change behavior without changing its base weights: instructions, demonstrations, retrieved records, low-rank adapters, inference procedures, and tools all intervene at different layers. This notebook asks what each intervention changes, what it costs, and what evidence supports the resulting answer.

You will build typed state, create a point-in-time retrieval packet, inspect a frozen model under evidence-position and few-shot interventions, require numeric claims to cite or refuse, audit a deterministic checker, inspect a low-rank update, and compare inference policies on a quality-cost-latency ledger.

By the end, you should be able to:

- draw the system boundary rather than evaluating only the generator;
- record the source, version, timestamp, and admissibility of retrieved context;
- distinguish prompting and retrieval from parameter adaptation;
- audit context sensitivity without changing the independent answer labels;
- calculate the parameter budget of a low-rank adaptation; and
- compare inference quality with tokens, calls, latency, tool use, and expected loss.

The context replay is a small pinned-model diagnostic, not a general effect estimate. The matrix and inference experiments are transparent teaching calibrations. They explain mechanisms and accounting; they do not benchmark a production language model.

### Course context

For a concise review and the optional textbook route, see [Week 11 summary](https://github.com/smc77/uc_finmlai/blob/main/lectures/week11/week11_summary.md) and [Week 11 reading guide](https://github.com/smc77/uc_finmlai/blob/main/lectures/week11/reading.md). The notebook itself is designed to remain understandable without them.

## How to use this notebook

At the first break, run any **Setup** cell and then the **Imports** cell. If there is no Setup cell, begin with Imports. After that, use the slide cue to jump to the named demo; you do not need to rerun the whole notebook at every break. To check the whole notebook from a clean start, use **Runtime → Run all**.

Each demo follows the same rhythm: predict what the output should show, run the cell, and follow the task immediately underneath it. An **After you run** cell is editable: double-click it, replace `[write here]`, and press Shift-Enter. When an editable research-record entry appears, its completed comparison sits below it in a collapsed box—write your version before opening that box. This notebook is generated from the same source code the lecturer runs live.

## Jump to a demonstration

Both links jump within *this* notebook — neither opens a new tab. Use **Colab** when the notebook is open in Colab, **Jupyter** when it is open in Jupyter or rendered to HTML. You do not need to scroll through or rerun the whole notebook.

> If the **Jupyter** links do nothing in JupyterLab or Notebook 7, that is the windowed renderer, not a broken link: cells outside the viewport are not in the page, so there is no anchor to scroll to. Set *Settings → Settings Editor → Notebook → Windowing mode* to `defer` (or `none`) and they work.

### Deck A

<ul>
<li>Demo 0 — Typed state, deterministic reduction, visible failure — <a href="#Demo-0-%E2%80%94-Typed-state,-deterministic-reduction,-visible-failure">Jupyter</a> · <a href="#scrollTo=md-fb0c7192ba32">Colab</a></li>
<li>Demo 1 — A point-in-time retrieval packet — <a href="#Demo-1-%E2%80%94-A-point-in-time-retrieval-packet">Jupyter</a> · <a href="#scrollTo=md-a412cec6da68">Colab</a></li>
<li>Demo 1a — A pinned generator with and without dated evidence — <a href="#Demo-1a-%E2%80%94-A-pinned-generator-with-and-without-dated-evidence">Jupyter</a> · <a href="#scrollTo=md-9b55b9dd3305">Colab</a></li>
<li>Demo 1b — Extraction proposal, trial ledger, and authority decision — <a href="#Demo-1b-%E2%80%94-Extraction-proposal,-trial-ledger,-and-authority-decision">Jupyter</a> · <a href="#scrollTo=md-a187cd983cf5">Colab</a></li>
</ul>

### Deck B

<ul>
<li>Demo 1c — Frozen-model context-position and few-shot replay — <a href="#Demo-1c-%E2%80%94-Frozen-model-context-position-and-few-shot-replay">Jupyter</a> · <a href="#scrollTo=md-34b4d0b2e148">Colab</a></li>
<li>Demo 2 — Cite or refuse as a deterministic numeric gate — <a href="#Demo-2-%E2%80%94-Cite-or-refuse-as-a-deterministic-numeric-gate">Jupyter</a> · <a href="#scrollTo=md-8318f8acd12c">Colab</a></li>
<li>Demo 3 — Audit the numeric gate on meaning-preserving and changing cases — <a href="#Demo-3-%E2%80%94-Audit-the-numeric-gate-on-meaning-preserving-and-changing-cases">Jupyter</a> · <a href="#scrollTo=md-f0cb8f24c56b">Colab</a></li>
<li>Demo 4 — A low-rank update trades parameters for approximation — <a href="#Demo-4-%E2%80%94-A-low-rank-update-trades-parameters-for-approximation">Jupyter</a> · <a href="#scrollTo=md-2ab7c993abe4">Colab</a></li>
<li>Demo 5 — Inference policies occupy a quality-cost-latency frontier — <a href="#Demo-5-%E2%80%94-Inference-policies-occupy-a-quality-cost-latency-frontier">Jupyter</a> · <a href="#scrollTo=md-61aa3067b49f">Colab</a></li>
</ul>

## Project research-record checkpoint — LLM system boundary

Keep the project evidence trail in the single project-root `RESEARCH_RECORD.md`; this notebook only identifies system fields that may now matter. Update the applicable fields:

1. **Layer audit:** objective, base model, prompt, context, adapter, tool, and output schema versions.
2. **Context packet:** decision time, source span, document version and hash, rank, admissibility, refusal.
3. **Context and claim audit:** frozen model and prompt, evidence-position intervention, independent labels, challenge categories, checker version, and confusion matrix.
4. **Adaptation choice:** prompting, retrieval, low-rank or full fine-tuning, training boundary, and simpler alternative.
5. **Inference budget:** calls, tokens, tools, router and argument errors, latency, cost convention, checker, and fallback.

Confirm that the frozen-plan commit still predates the final lockbox. Separate what the test established from what remains untested, and log every post-freeze change.

### Imports

> **Run this once before any demo.** It loads the packages and helper functions used below; there is no result to interpret in this cell.

In [ ]:
import hashlib
import json
import re
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass
from pathlib import Path
from urllib.request import urlopen

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.metrics.pairwise import cosine_similarity

rng = np.random.default_rng(42)


def finish_plot():
    """Show figures interactively and close them in noninteractive checks."""
    if "agg" not in plt.get_backend().lower():
        plt.show()
    plt.close()


def save_demo_plot(fig, filename):
    """Persist the canonical lecture figure when running inside the course repo."""
    candidates = [
        Path("lectures/week11/figures"),
        Path("../figures"),
    ]
    target_dir = next((path for path in candidates if path.exists()), None)
    if target_dir is not None:
        fig.savefig(target_dir / filename, dpi=190, bbox_inches="tight",
                    facecolor="white")

### Demo 0 — Typed state, deterministic reduction, visible failure

**Plain-language question:** What should a larger AI application do when two components finish successfully but a required safety check does not return?

**Scenario.** An analyst's application sends the same request to three small workers. One reads a filing, one reads a market-price record, and one checks the action policy. The policy worker is deliberately made to time out. This is a controlled failure, not an accidental notebook error.

**Predict before running.** The filing and market findings should survive; the policy timeout should be recorded visibly; research may continue with the two findings; and an external action should remain blocked because its policy check never completed.

**How to read the output.** `What each worker returned` shows which worker completed or failed. `Findings that survived the reduction` contains only evidence returned by completed workers. The separate research and action statuses demonstrate that the same partial state can be adequate for analysis but inadequate for a higher-consequence decision.

> **Boundary of the demonstration:** No LLM is called, and nothing is being predicted. The demo isolates a system-design principle: pass structured results between components, preserve failures as data, and let an explicit rule—not a fluent model—decide what the incomplete result is permitted to support.

In [ ]:
# Controlled scenario: the filing and market workers succeed, while the policy
# worker times out. The point is to observe how explicit state and permissions
# behave under a partial failure; this demo does not call an LLM.
@dataclass(frozen=True)
class TaskPacket:
    request_id: str
    decision_day: int
    documents: tuple


@dataclass(frozen=True)
class Finding:
    worker: str
    claim: str
    source: str
    available_day: int


@dataclass(frozen=True)
class NodeResult:
    worker: str
    status: str
    findings: tuple = ()
    error: str = ""


graph_task = TaskPacket(
    request_id="issuer-17-day-20",
    decision_day=20,
    documents=(
        {"domain": "filing", "source": "10-Q-A", "available_day": 10,
         "claim": "revenue was 4.2 billion"},
        {"domain": "market", "source": "PX-17", "available_day": 20,
         "claim": "shares closed at 31.40"},
        {"domain": "policy", "source": "POL-v3", "available_day": 5,
         "claim": "manual review above 4.0 leverage"},
    ),
)


def graph_worker(worker, task, fail=False):
    """Return one typed result; never pass an arbitrary chat transcript."""
    if fail:
        raise TimeoutError("declared worker timeout")
    rows = [row for row in task.documents if row["domain"] == worker]
    findings = tuple(
        Finding(worker, row["claim"], row["source"], row["available_day"])
        for row in rows
    )
    return NodeResult(worker=worker, status="complete", findings=findings)


def run_node(worker, task, fail=False):
    try:
        return graph_worker(worker, task, fail=fail)
    except Exception as exc:
        return NodeResult(worker=worker, status="failed", error=type(exc).__name__)


worker_plan = [("filing", False), ("market", False), ("policy", True)]
with ThreadPoolExecutor(max_workers=3) as pool:
    node_results = list(pool.map(
        lambda spec: run_node(spec[0], graph_task, spec[1]), worker_plan
    ))

# Deterministic reducer: stable order, explicit coverage, no model required.
node_results = sorted(node_results, key=lambda result: result.worker)
reduced = sorted(
    (finding for result in node_results for finding in result.findings),
    key=lambda finding: (finding.available_day, finding.worker, finding.source),
)
completed = {result.worker for result in node_results if result.status == "complete"}
research_required = {"filing", "market"}
action_required = {"filing", "market", "policy"}
admissible = all(f.available_day <= graph_task.decision_day for f in reduced)
research_status = (
    "usable_partial"
    if research_required <= completed and admissible
    else "blocked"
)
action_status = (
    "permitted"
    if action_required <= completed and admissible
    else "blocked"
)

print("\nSCENARIO")
print("  Filing and market evidence return; the required policy check times out.")
print("\n1) TYPED HANDOFFS")
print("  request -> workers : TaskPacket")
print("  workers -> reducer : NodeResult[Finding]")
print("  reducer -> verifier: ordered Finding[] + coverage")
print("\n2) WHAT EACH WORKER RETURNED")
for result in node_results:
    print(f"  {result.worker:<7} {result.status:<8} {result.error or 'ok'}")
print("\n3) FINDINGS THAT SURVIVED THE REDUCTION")
for finding in reduced:
    print(f"  day {finding.available_day:>2} [{finding.source}] {finding.claim}")
print("\n4) WHAT THIS INCOMPLETE RESULT MAY SUPPORT")
print(f"  Coverage: {len(completed)}/{len(worker_plan)} workers completed")
print(f"  Research status: {research_status} (filing + market are sufficient)")
print(f"  Action status: {action_status} (policy approval is also required)")
print("\nTAKEAWAY")
print("  Preserve the timeout as visible state. Keep the valid evidence, but do not")
print("  let a partial result cross an authority boundary that requires policy review.")

### Mini-project: audit the decoder and system

> **Editable lecture task.** Complete this cell after running the named demo. The prompt mirrors the lecture mini-project even when the required work is a written classification, audit, diagram, or decision rather than additional code.

**Notebook:** run **Demo 0 — Typed state and visible failure**.

Using the chapter figure and one LLM application:

1. name its decoder objective;
2. list every system layer around it;
3. draw the smallest dependency graph that performs the task;
4. name the state object crossing every edge;
5. locate the evidence and authority refusal points.

**Research record:** objective; system layers and versions; evidence-existence
check; edge-state schema; refusal points; authority owner.

*Short on time?* Finish the ledger and both refusal points first.

#### Your response

Replace the placeholders below with the requested output before opening the lecture debrief.

- **Result or artifact:** `[write here]`
- **Research-record fields:** `[write here]`
- **Bounded conclusion:** `[write here]`

### Demo 1 — A point-in-time retrieval packet

**Plain-language question:** If an analyst asks a question on day 20, what evidence may the system show the model—especially when a corrected filing arrives later?

**Scenario.** Four original filing records become public on day 10. An amendment on day 40 changes reported revenue from $4.2 billion to $3.8 billion. The system must answer as of the declared decision day rather than silently using the latest version.

**Predict before running.** You should see three distinct checks:

1. For an operating-margin question on day 20, retrieval ranks only records that existed by day 20 and preserves their source, version, span, hash, rank, and score.
2. A revenue query uses the original $4.2 billion record on day 20 but the $3.8 billion amendment on day 50. The amendment cannot travel backward in time.
3. A headcount query refuses because the available packet contains no headcount record. Missing evidence is not permission to invent a value.

**How the three checks fit together.** Availability filtering determines which records are eligible; retrieval ranks the eligible records; and the packet preserves enough provenance to audit the result later. Demo 1a then asks a separate question: what does an actual generator do when it receives—or does not receive—that packet?

> **Boundary of the demonstration:** This cell uses a small synthetic fixture and a deterministic TF–IDF retriever. It demonstrates the packet contract and historical clock, not production retrieval quality and not LLM accuracy.

In [ ]:
# Controlled scenario: original records appear on day 10 and a revenue amendment
# appears on day 40. The demo separates eligibility, ranking, version choice, and
# refusal so that a later correction cannot leak into an earlier decision.
records = [
    {
        "source": "A1",
        "version": "1.0",
        "supersedes": None,
        "field": "operating_margin",
        "available_day": 10,
        "span": "p. 2, paragraph 1",
        "text": "Operating margin expanded 220 basis points to 28.4%.",
    },
    {
        "source": "A2",
        "version": "1.0",
        "supersedes": None,
        "field": "revenue",
        "available_day": 10,
        "span": "p. 2, paragraph 2",
        "text": "Revenue was $4.2 billion in the quarter.",
    },
    {
        "source": "A3",
        "version": "1.0",
        "supersedes": None,
        "field": "free_cash_flow",
        "available_day": 10,
        "span": "p. 3, paragraph 1",
        "text": "Free cash flow was $0.9 billion, up 12%.",
    },
    {
        "source": "A4",
        "version": "1.0",
        "supersedes": None,
        "field": "dividend",
        "available_day": 10,
        "span": "p. 4, paragraph 2",
        "text": "The board declared a dividend of $0.42 per share.",
    },
    {
        "source": "B1",
        "version": "1.1",
        "supersedes": "A2",
        "field": "revenue",
        "available_day": 40,
        "span": "p. 1, paragraph 1",
        "text": "Amended revenue for the quarter was $3.8 billion.",
    },
]

for record in records:
    record["document_hash"] = hashlib.sha256(
        record["text"].encode("utf-8")
    ).hexdigest()[:12]


def retrieve(question, decision_day, k=3):
    """Retrieve only records available by the declared decision day."""
    admissible = [r for r in records if r["available_day"] <= decision_day]
    superseded = {
        row["supersedes"] for row in admissible if row["supersedes"] is not None
    }
    admissible = [row for row in admissible if row["source"] not in superseded]
    texts = [r["text"] for r in admissible]
    vectorizer = TfidfVectorizer().fit(texts)
    doc_vectors = vectorizer.transform(texts)
    query_vector = vectorizer.transform([question])
    scores = cosine_similarity(query_vector, doc_vectors).ravel()
    order = np.argsort(-scores)[:k]
    return [
        {**admissible[i], "rank": rank, "score": scores[i],
         "index_version": "week11-fixture-v1"}
        for rank, i in enumerate(order, start=1)
    ]


question = "what did the filing report about operating margin"
decision_day = 20
packet = retrieve(question, decision_day)

print("\nSCENARIO")
print("  Build the evidence packet that an analyst was allowed to use on day 20.")
print("\n1) RETRIEVE ONLY RECORDS AVAILABLE BY THE DECISION DAY")
print(f"  Question: {question!r}")
print(f"  Decision day: {decision_day}")
for row in packet:
    print(f"  rank={row['rank']} [{row['source']} v{row['version']}] "
          f"available={row['available_day']:>2} span={row['span']} "
          f"hash={row['document_hash']} score={row['score']:.3f}: {row['text']}")

# Version-clock check: the amendment cannot travel backward.
print("\n2) APPLY THE VERSION CLOCK BEFORE CHOOSING A REVENUE RECORD")
for day in (20, 50):
    revenue_packet = retrieve("what was quarterly revenue", day, k=1)
    print(f"  Revenue source at day {day}: {revenue_packet[0]['source']} "
          f"-> {revenue_packet[0]['text']}")


def resolve_field(field, decision_day):
    """Return the eligible field record or an explicit missing-evidence refusal."""
    admissible = [
        row for row in records
        if row["field"] == field and row["available_day"] <= decision_day
    ]
    superseded = {
        row["supersedes"] for row in admissible if row["supersedes"] is not None
    }
    admissible = [row for row in admissible if row["source"] not in superseded]
    if not admissible:
        return {
            "status": "refuse",
            "reason": f"no admissible {field} record by day {decision_day}",
        }
    return {"status": "answer", "record": admissible[-1]}


print("\n3) REFUSE WHEN THE ELIGIBLE PACKET HAS NO REQUESTED FIELD")
print("  Headcount query:", resolve_field("headcount", decision_day=20))
print("\nTAKEAWAY")
print("  First filter by historical availability; then rank. Preserve provenance,")
print("  use amendments only after they arrive, and refuse fields the packet lacks.")

### Demo 1a — A pinned generator with and without dated evidence

> **Demonstration:** State what you expect before running the cell; afterward, explain which output supports or contradicts that expectation.

In [ ]:
fixture_name = "week11_llm_generation_fixture.json"
fixture_candidates = [
    Path(fixture_name),
    Path("lectures/week11") / fixture_name,
    Path("data/model_fixtures") / fixture_name,
    Path("../../../data/model_fixtures") / fixture_name,
]
fixture_path = next((path for path in fixture_candidates if path.exists()), None)
fixture_url = (
    "https://raw.githubusercontent.com/smc77/uc_finmlai/"
    f"main/lectures/week11/{fixture_name}"
)
fixture_sha256 = "8f3ecf97fc3b336d56b7f5c11ed6d4bce45c90942b6e7b383b406d162dd1fa64"
if fixture_path is not None:
    fixture_bytes = fixture_path.read_bytes()
    fixture_location = str(fixture_path)
else:
    print("Local fixture not found; downloading the pinned public Week 11 replay.")
    with urlopen(fixture_url, timeout=30) as response:
        fixture_bytes = response.read()
    fixture_location = fixture_url

actual_fixture_sha256 = hashlib.sha256(fixture_bytes).hexdigest()
if actual_fixture_sha256 != fixture_sha256:
    raise RuntimeError(
        "Week 11 fixture hash mismatch: "
        f"expected {fixture_sha256}, received {actual_fixture_sha256}"
    )
llm_fixture = json.loads(fixture_bytes.decode("utf-8"))
print(f"Loaded pinned fixture from {fixture_location}")
fixture_sources = {row["source"] for row in llm_fixture["evidence"]}
fixture_numbers = set(re.findall(
    r"\d+\.?\d*", " ".join(row["text"] for row in llm_fixture["evidence"])
))

llm_rows = []
for run in llm_fixture["runs"]:
    output = run["output"]
    output_numbers = set(re.findall(r"\d+\.?\d*", output))
    refused = "REFUSE" in output.upper()
    supported = (
        run["case_id"] == "answerable"
        and {"4.2", "28.4"} <= output_numbers
        and output_numbers <= fixture_numbers
    )
    cited_sources = set(re.findall(r"\[([A-Z]\d+)\]", output))
    citations_resolve = bool(cited_sources) and cited_sources <= fixture_sources
    exact_refusal = output.strip() == "REFUSE: unsupported by supplied evidence."
    policy_compliant = (
        supported and citations_resolve
        if run["case_id"] == "answerable"
        else exact_refusal
    )
    llm_rows.append({
        "case": run["case_id"],
        "configuration": run["configuration"],
        "supported": supported,
        "citations_resolve": citations_resolve,
        "refused": refused,
        "format_ok": exact_refusal if run["case_id"] == "unanswerable" else True,
        "policy_compliant": policy_compliant,
        "output": output,
    })

llm_audit = pd.DataFrame(llm_rows)
print(f"\nPinned LLM fixture: {llm_fixture['model']} @ "
      f"{llm_fixture['revision'][:8]}")
for row in llm_rows:
    print(f"\n[{row['case']} / {row['configuration']}]\n{row['output']}")
print("\nClaim, citation, and refusal audit:")
print(llm_audit[["case", "configuration", "supported",
                 "citations_resolve", "refused", "format_ok",
                 "policy_compliant"]])
print("This designed four-run fixture demonstrates behavior; it is not an error-rate estimate.")

### Mini-project: assemble a context packet

> **Editable lecture task.** Complete this cell after running the named demo. The prompt mirrors the lecture mini-project even when the required work is a written classification, audit, diagram, or decision rather than additional code.

**Notebook:** run **Demo 1 — A point-in-time retrieval packet**, then inspect
the guided **Demo 1a — A pinned generator with and without dated evidence**.

For one filing question:

1. set a decision timestamp;
2. retrieve three admissible passages;
3. preserve source spans, versions, hashes, ranks, and scores;
4. preserve the absent-field refusal; and
5. compare the deterministic contract with Demo 1a's generator behavior.

**Research record:** decision time; source span; document version and hash;
retrieval rank; admissibility rule; support or refusal.

#### Your response

Replace the placeholders below with the requested output before opening the lecture debrief.

- **Result or artifact:** `[write here]`
- **Research-record fields:** `[write here]`
- **Bounded conclusion:** `[write here]`

### Demo 1b — Extraction proposal, trial ledger, and authority decision

> **Break cue:** Deck A, after recording segment S3. Read the task on the preceding slide, predict what the output should show, then run the cell. Record what the task asks for: the specific values you observed, the sample and settings that produced them, and one conclusion the result cannot support.

In [ ]:
extraction_source = {
    "source": "G1",
    "version": "1.0",
    "available_day": 12,
    "span": "p. 6, paragraph 2",
    "text": (
        "For FY2026, management raised expected revenue growth "
        "from 8% to 10%."
    ),
}
extraction_source["document_hash"] = hashlib.sha256(
    extraction_source["text"].encode("utf-8")
).hexdigest()[:12]

trial_ledger = pd.DataFrame([
    ("lexical-v1", "guidance-schema-v1", 0, 0.68),
    ("guidance-v1", "guidance-schema-v1", 1, 0.79),
    ("guidance-v2", "guidance-schema-v1", 0, 0.84),
], columns=["procedure", "schema", "retries", "validation_score"])
selected_trial = trial_ledger.sort_values(
    ["validation_score", "procedure"], ascending=[False, True]
).iloc[0]

guidance_candidate = {
    "metric": "revenue_growth",
    "direction": "raised",
    "from_percent": 8.0,
    "to_percent": 10.0,
    "period": "FY2026",
    "source": extraction_source["source"],
    "span": extraction_source["span"],
    "document_hash": extraction_source["document_hash"],
    "schema": selected_trial["schema"],
}
candidate_supported = all(
    token in extraction_source["text"]
    for token in ("FY2026", "raised", "8%", "10%")
)

tool_proposal = {
    "tool": "write_research_database",
    "record_key": "issuer-17-FY2026-revenue-growth",
    "value": guidance_candidate,
    "approved": False,
    "idempotency_key": "guidance-G1-v1-issuer17-FY2026",
}
allowed_tools = {"write_research_database"}
authority_decision = (
    "permit"
    if tool_proposal["tool"] in allowed_tools and tool_proposal["approved"]
    else "block"
)

print("\nExtraction evidence:")
print(f"  day {extraction_source['available_day']} "
      f"[{extraction_source['source']} v{extraction_source['version']}] "
      f"span={extraction_source['span']} "
      f"hash={extraction_source['document_hash']}")
print("Typed guidance candidate:")
print(pd.Series(guidance_candidate))
print(f"Support check: {'passed' if candidate_supported else 'failed'}")
print("Trial ledger:")
print(trial_ledger)
print(f"Selected on validation: {selected_trial['procedure']} "
      f"at {selected_trial['validation_score']:.2f} from {len(trial_ledger)} trials")
print("Tool proposal:")
print(pd.Series(tool_proposal).drop("value"))
print(f"Authority decision: {authority_decision}")
print("A supported candidate can enter review; it cannot approve its own write.")

### Mini-project: audit one extraction proposal

> **Editable lecture task.** Complete this cell after running the named demo. The prompt mirrors the lecture mini-project even when the required work is a written classification, audit, diagram, or decision rather than additional code.

**Notebook:** run **Demo 1b — Extraction proposal, trial ledger, and authority decision**.

1. inspect the typed guidance candidate and its support check;
2. preserve all three compared extraction procedures and the selection rule;
3. inspect the proposed database write and independent approval bit; and
4. record the idempotency key that the executor would have to enforce.

**Research record:** source span, version, and hash; prompt and schema; trial
count; validation boundary; proposal; approval bit; executor; idempotency key.

#### Your response

Replace the placeholders below with the requested output before opening the lecture debrief.

- **Result or artifact:** `[write here]`
- **Research-record fields:** `[write here]`
- **Bounded conclusion:** `[write here]`

### Demo 1c — Frozen-model context-position and few-shot replay

> **Demonstration:** State what you expect before running the cell; afterward, explain which output supports or contradicts that expectation.

In [ ]:
context_replay = pd.DataFrame(llm_fixture["context_experiment"]["runs"])
if "distractor_order" not in context_replay:
    context_replay["distractor_order"] = "canonical"
context_summary = (
    context_replay
    .groupby(["distractor_order", "shot", "position"], as_index=False)
    .agg(
        answer_accuracy=("answer_correct", "mean"),
        citation_accuracy=("citation_correct", "mean"),
        mean_input_tokens=("input_tokens", "mean"),
        cases=("case_id", "count"),
    )
)
shot_order = ["zero-shot", "few-shot"]
position_order = ["first", "middle", "last"]
distractor_orders = llm_fixture["context_experiment"]["design"].get(
    "distractor_orders", ["canonical"]
)
context_summary["shot"] = pd.Categorical(
    context_summary["shot"], categories=shot_order, ordered=True
)
context_summary["position"] = pd.Categorical(
    context_summary["position"], categories=position_order, ordered=True
)
context_summary = context_summary.sort_values(
    ["distractor_order", "shot", "position"]
)
print("\nFrozen-model context sensitivity (12 paired questions per cell):")
print(context_summary.to_string(index=False, formatters={
    "answer_accuracy": "{:.3f}".format,
    "citation_accuracy": "{:.3f}".format,
    "mean_input_tokens": "{:.1f}".format,
}))

for distractor_order in distractor_orders:
    few = (
        context_replay[
            (context_replay["distractor_order"] == distractor_order)
            & (context_replay["shot"] == "few-shot")
        ]
        .pivot(index="case_id", columns="position", values="answer_correct")
        .astype(bool)
    )
    paired = {
        "first only": int((few["first"] & ~few["middle"]).sum()),
        "middle only": int((~few["first"] & few["middle"]).sum()),
        "both": int((few["first"] & few["middle"]).sum()),
        "neither": int((~few["first"] & ~few["middle"]).sum()),
    }
    print(f"Few-shot first versus middle, {distractor_order}: {paired}")

fig, axes = plt.subplots(
    len(distractor_orders), 2,
    figsize=(9.4, 3.35 * len(distractor_orders)),
    sharey=True,
    squeeze=False,
)
x = np.arange(len(position_order))
width = 0.34
colors = {"zero-shot": "#6b7280", "few-shot": "#1f3b70"}
for row_index, distractor_order in enumerate(distractor_orders):
    order_rows = context_summary[
        context_summary["distractor_order"] == distractor_order
    ]
    for column_index, (metric, title) in enumerate([
        ("answer_accuracy", "answer accuracy"),
        ("citation_accuracy", "citation-ID accuracy"),
    ]):
        ax = axes[row_index, column_index]
        for offset, shot in enumerate(shot_order):
            rows = order_rows[order_rows["shot"] == shot]
            delta = (offset - 0.5) * width
            bars = ax.bar(
                x + delta, rows[metric], width=width,
                label=shot, color=colors[shot], alpha=0.92,
            )
            for bar, value, cases in zip(bars, rows[metric], rows["cases"]):
                ax.text(
                    bar.get_x() + bar.get_width() / 2, value + 0.025,
                    f"{int(round(value * cases))}/{int(cases)}",
                    ha="center", fontsize=7.5,
                )
        ax.set_title(f"{distractor_order} order · {title}")
        ax.set_xticks(x, position_order)
        ax.set_ylim(0, 1.09)
        ax.grid(axis="y", alpha=0.22)
        ax.set_xlabel("position of the relevant record")
        if column_index == 0:
            ax.set_ylabel("share correct")
        if row_index == 0 and column_index == 1:
            ax.legend(frameon=False, loc="upper right")
fig.tight_layout()
save_demo_plot(fig, "demo1c_context_sensitivity.png")
finish_plot()

### Demo 2 — Cite or refuse as a deterministic numeric gate

> **Break cue:** Deck B, after recording segment S4. Read the task on the preceding slide, predict what the output should show, then run the cell. Record what the task asks for: the specific values you observed, the sample and settings that produced them, and one conclusion the result cannot support.

In [ ]:
NUMBER = re.compile(r"\d+\.?\d*")


def numbers_in(text):
    return set(NUMBER.findall(text))


def cite_or_refuse(answer_text, context_packet):
    """Reject a numeric claim when its figures are absent from context."""
    answer_numbers = numbers_in(answer_text)
    context_numbers = set().union(
        *(numbers_in(row["text"]) for row in context_packet)
    )
    missing = answer_numbers - context_numbers
    if missing:
        return {
            "status": "refuse",
            "answer": None,
            "missing": sorted(missing),
            "sources": [],
        }
    return {
        "status": "answer",
        "answer": answer_text,
        "missing": [],
        "sources": [row["source"] for row in context_packet],
    }


grounded = "Operating margin expanded 220 basis points to 28.4%."
fabricated = "Operating margin reached 31.2%."
print("\nGrounded candidate:", cite_or_refuse(grounded, packet))
print("Fabricated candidate:", cite_or_refuse(fabricated, packet))
print("Passing this gate means only that the numbers appeared in context.")

### Demo 3 — Audit the numeric gate on meaning-preserving and changing cases

> **Break cue:** Deck B, after recording segment S4. Read the task on the preceding slide, predict what the output should show, then run the cell. Record what the task asks for: the specific values you observed, the sample and settings that produced them, and one conclusion the result cannot support.

In [ ]:
challenge_set = [
    # category, claim, context, true support
    ("exact", "Margin was 28.4%.", "Margin was 28.4%.", 1),
    ("exact", "Free cash flow was 4.1.", "Free cash flow was 4.1.", 1),
    ("equivalent", "Margin was 28.4%.", "Margin was 0.284.", 1),
    ("equivalent", "Revenue was 4200 million.", "Revenue was 4.2 billion.", 1),
    ("absent", "Revenue was 31.2.", "Revenue was 12.0.", 0),
    ("absent", "EPS was 3.40.", "EPS was 2.15.", 0),
    ("wrong period", "2024 revenue was 4.2.", "2023 revenue was 4.2.", 0),
    ("wrong period", "2025 margin was 28.4.", "2024 margin was 28.4.", 0),
    ("negation", "Margin rose 8%.", "Margin did not rise 8%.", 0),
    ("negation", "Debt fell 4.2.", "Debt did not fall 4.2.", 0),
    ("wrong metric", "Revenue was 4.2.", "Debt was 4.2.", 0),
    ("wrong metric", "EPS was 2.15.", "Dividend was 2.15.", 0),
]


def numeric_overlap_support(claim, context):
    """Baseline only: every claim number must occur literally in context."""
    return int(numbers_in(claim).issubset(numbers_in(context)))


truth = np.array([row[3] for row in challenge_set])
pred = np.array([numeric_overlap_support(row[1], row[2])
                 for row in challenge_set])
audit = pd.DataFrame(challenge_set,
                     columns=["category", "claim", "context", "supported"])
audit["predicted"] = pred
audit["correct"] = audit["supported"] == audit["predicted"]

print("\nNumeric-overlap challenge audit:")
print(audit[["category", "supported", "predicted", "correct"]])
print(f"Accuracy: {accuracy_score(truth, pred):.3f}")
print("Confusion matrix [[TN, FP], [FN, TP]]:")
print(confusion_matrix(truth, pred))
# Expected: the gate handles exact matches and absent claims correctly but fails
#           equivalent restatements. That failure is the point of the audit.

### Mini-project: challenge the claim screen

> **Editable lecture task.** Complete this cell after running the named demo. The prompt mirrors the lecture mini-project even when the required work is a written classification, audit, diagram, or decision rather than additional code.

**Notebook:** run **Demos 2–3 — Numeric gate and challenge audit**.

1. Preserve the grounded answer and fabricated-value refusal from Demo 2.
2. Reproduce Demo 3's confusion matrix on twelve controlled pairs.
3. Explain one false support and one missed support.
4. Add one case where changing context order should not change the correct label.

**Research record:** permitted packet; atomic claim; independent label; checker
version; confusion matrix; context-order invariant; boundary of the screen.

#### Your response

Replace the placeholders below with the requested output before opening the lecture debrief.

- **Result or artifact:** `[write here]`
- **Research-record fields:** `[write here]`
- **Bounded conclusion:** `[write here]`

### Demo 4 — A low-rank update trades parameters for approximation

> **Break cue:** Deck B, after recording segment S5. Read the task on the preceding slide, predict what the output should show, then run the cell. Record what the task asks for: the specific values you observed, the sample and settings that produced them, and one conclusion the result cannot support.

In [ ]:
# This is a matrix-mechanism illustration, not an LLM quality benchmark.
lora_rng = np.random.default_rng(11)
d_in = d_out = 128
true_rank = 6
left = lora_rng.normal(size=(d_out, true_rank))
right = lora_rng.normal(size=(true_rank, d_in))
target_delta = left @ right + 0.20 * lora_rng.normal(size=(d_out, d_in))
u, singular_values, vt = np.linalg.svd(target_delta, full_matrices=False)

lora_rows = []
for rank in (1, 2, 4, 6, 8, 16, 32):
    approximation = (
        u[:, :rank] * singular_values[:rank]
    ) @ vt[:rank, :]
    relative_error = (
        np.linalg.norm(target_delta - approximation, ord="fro")
        / np.linalg.norm(target_delta, ord="fro")
    )
    trainable = rank * (d_in + d_out)
    lora_rows.append({
        "rank": rank,
        "trainable_parameters": trainable,
        "percent_of_full_matrix": 100 * trainable / (d_in * d_out),
        "relative_update_error": relative_error,
    })

lora_audit = pd.DataFrame(lora_rows)
print("\nLow-rank update audit:")
print(lora_audit.to_string(index=False, formatters={
    "percent_of_full_matrix": "{:.2f}".format,
    "relative_update_error": "{:.3f}".format,
}))
under_ten_percent = lora_audit[
    lora_audit["percent_of_full_matrix"] <= 10
]
selected_lora = under_ten_percent.sort_values(
    ["relative_update_error", "rank"]
).iloc[0]
print(
    f"Best listed rank under a 10% parameter budget: r={int(selected_lora['rank'])}; "
    f"relative error={selected_lora['relative_update_error']:.3f}."
)
print(
    "The planted update is approximately rank six; a real model's useful update "
    "rank and task quality must be measured rather than inferred from this matrix."
)

fig, ax1 = plt.subplots(figsize=(8.8, 4.4))
ax2 = ax1.twinx()
ax1.plot(lora_audit["rank"], lora_audit["relative_update_error"],
         marker="o", color="#1f3b70", linewidth=2.2, label="relative error")
ax2.plot(lora_audit["rank"], lora_audit["percent_of_full_matrix"],
         marker="s", color="#a32121", linewidth=2.0, label="parameter share")
ax1.axvline(true_rank, color="#2e7d32", linestyle="--", linewidth=1.6,
            label="planted rank")
ax1.set_xlabel("adapter rank")
ax1.set_ylabel("relative update error", color="#1f3b70")
ax2.set_ylabel("trainable parameters (% of full matrix)", color="#a32121")
ax1.set_title("Low rank reduces error while increasing the trainable budget")
ax1.grid(alpha=0.2)
lines = ax1.get_lines() + ax2.get_lines()
ax1.legend(lines, [line.get_label() for line in lines], loc="center right")
fig.tight_layout()
save_demo_plot(fig, "demo4_lora_tradeoff.png")
finish_plot()

### Mini-project: audit the LoRA budget

> **Editable lecture task.** Complete this cell after running the named demo. The prompt mirrors the lecture mini-project even when the required work is a written classification, audit, diagram, or decision rather than additional code.

**Notebook:** **Demo 4 is complete; do not rerun it.** Audit the saved
`lora_audit` table and generated tradeoff figure.

1. Verify the trainable-parameter count at ranks 1, 4, 6, and 16.
2. Identify the best listed rank under a 10% parameter budget.
3. Compare its update error with the next larger rank.
4. State what evidence would be required before selecting a language-model adapter.

**Research record:** matrix dimensions; candidate ranks; parameter-count rule;
relative-error rule; selected rank; budget; boundary of the illustration.

#### Your response

Replace the placeholders below with the requested output before opening the lecture debrief.

- **Result or artifact:** `[write here]`
- **Research-record fields:** `[write here]`
- **Bounded conclusion:** `[write here]`

### Demo 5 — Inference policies occupy a quality-cost-latency frontier

> **Break cue:** Deck B, after recording segment S6. Read the task on the preceding slide, predict what the output should show, then run the cell. Record what the task asks for: the specific values you observed, the sample and settings that produced them, and one conclusion the result cannot support.

In [ ]:
# Known-truth teaching simulation. The planted probabilities and prices are not
# measurements of any provider, model, or financial deployment.
budget_rng = np.random.default_rng(19)
n_cases = 4000
difficulty = budget_rng.normal(size=n_cases)
shared_noise = budget_rng.normal(scale=0.55, size=n_cases)


def correctness(base_logit, independent_scale=0.65):
    latent = base_logit - 0.80 * difficulty + shared_noise
    latent += budget_rng.normal(scale=independent_scale, size=n_cases)
    probability = 1 / (1 + np.exp(-latent))
    return budget_rng.binomial(1, probability)


direct_ok = correctness(1.05)
depth_ok = correctness(1.55, independent_scale=0.50)
breadth_draws = np.column_stack([correctness(1.05) for _ in range(5)])
breadth_ok = (breadth_draws.sum(axis=1) >= 3).astype(int)
tool_needed = budget_rng.binomial(1, 0.40, n_cases)
tool_fallback_ok = correctness(1.15)
routing_rng = np.random.default_rng(1919)
router_draw = routing_rng.random(n_cases)
argument_draw = routing_rng.random(n_cases)
router_miss_rate = 0.05
argument_error_rate = 0.02
recognized_tool_case = (tool_needed == 1) & (router_draw >= router_miss_rate)
valid_tool_call = recognized_tool_case & (argument_draw >= argument_error_rate)
tool_ok = np.where(
    tool_needed == 0,
    tool_fallback_ok,
    np.where(valid_tool_call, 1, direct_ok),
)
realized_tool_calls = recognized_tool_case.mean()

# Illustrative accounting prices, explicitly declared rather than vendor quotes.
input_price_per_million = 2.00
output_price_per_million = 8.00
tool_call_price = 0.0002
wrong_answer_loss = 0.08

policy_spec = [
    ("direct", direct_ok, 500, 120, 1, 0.0, 0.8),
    ("deeper reasoning", depth_ok, 500, 620, 1, 0.0, 2.4),
    ("five samples + vote", breadth_ok, 2500, 600, 5, 0.0, 1.5),
    ("tool route (5% miss)", tool_ok, 560, 170, 1, realized_tool_calls, 1.1),
]

budget_rows = []
for name, correct, input_tokens, output_tokens, calls, tools, latency in policy_spec:
    inference_cost = (
        input_tokens * input_price_per_million / 1_000_000
        + output_tokens * output_price_per_million / 1_000_000
        + tools * tool_call_price
    )
    error_rate = 1 - correct.mean()
    expected_total_loss = inference_cost + wrong_answer_loss * error_rate
    budget_rows.append({
        "policy": name,
        "accuracy": correct.mean(),
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "calls": calls,
        "tool_calls": tools,
        "illustrative_cost_dollars": inference_cost,
        "illustrative_latency_seconds": latency,
        "expected_total_loss": expected_total_loss,
    })

inference_frontier = pd.DataFrame(budget_rows)
print("\nInference policy ledger:")
print(inference_frontier.to_string(index=False, formatters={
    "accuracy": "{:.3f}".format,
    "illustrative_cost_dollars": "{:.5f}".format,
    "illustrative_latency_seconds": "{:.1f}".format,
    "expected_total_loss": "{:.4f}".format,
}))
selected_policy = inference_frontier.sort_values(
    ["expected_total_loss", "illustrative_latency_seconds"]
).iloc[0]
print(
    f"Lowest expected total loss at ${wrong_answer_loss:.2f} per wrong answer: "
    f"{selected_policy['policy']}."
)
print(
    "Changing the value of an error, latency limit, token prices, or planted "
    "accuracy can change the selected policy."
)

route_sensitivity_rows = []
for miss_rate in (0.00, 0.05, 0.10):
    recognized = (tool_needed == 1) & (router_draw >= miss_rate)
    valid_call = recognized & (argument_draw >= argument_error_rate)
    routed_ok = np.where(
        tool_needed == 0,
        tool_fallback_ok,
        np.where(valid_call, 1, direct_ok),
    )
    tool_share = recognized.mean()
    route_cost = (
        560 * input_price_per_million / 1_000_000
        + 170 * output_price_per_million / 1_000_000
        + tool_share * tool_call_price
    )
    route_sensitivity_rows.append({
        "router_miss_rate": miss_rate,
        "invalid_argument_rate": argument_error_rate,
        "accuracy": routed_ok.mean(),
        "tool_calls_per_question": tool_share,
        "expected_total_loss": route_cost + wrong_answer_loss * (1 - routed_ok.mean()),
    })
route_sensitivity = pd.DataFrame(route_sensitivity_rows)
print("\nTool-route sensitivity:")
print(route_sensitivity.to_string(index=False, formatters={
    "router_miss_rate": "{:.0%}".format,
    "invalid_argument_rate": "{:.0%}".format,
    "accuracy": "{:.3f}".format,
    "tool_calls_per_question": "{:.3f}".format,
    "expected_total_loss": "{:.4f}".format,
}))

fig, ax = plt.subplots(figsize=(8.8, 4.5))
sizes = 140 + 75 * inference_frontier["illustrative_latency_seconds"]
scatter = ax.scatter(
    inference_frontier["illustrative_cost_dollars"],
    inference_frontier["accuracy"],
    s=sizes,
    c=["#6b7280", "#1f3b70", "#a32121", "#2e7d32"],
    alpha=0.88,
)
for _, row in inference_frontier.iterrows():
    ax.annotate(row["policy"],
                (row["illustrative_cost_dollars"], row["accuracy"]),
                xytext=(6, 6), textcoords="offset points", fontsize=8.5)
ax.set_xlabel("illustrative inference cost per question ($)")
ax.set_ylabel("accuracy in the known-truth simulation")
ax.set_title("Inference policies trade quality, cost, and latency")
ax.grid(alpha=0.2)
ax.text(0.99, 0.02, "larger marker = longer latency",
        transform=ax.transAxes, ha="right", va="bottom", fontsize=9,
        color="#4b5563")
fig.tight_layout()
save_demo_plot(fig, "demo5_inference_frontier.png")
finish_plot()

### Mini-project: choose an inference policy

> **Editable lecture task.** Complete this cell after running the named demo. The prompt mirrors the lecture mini-project even when the required work is a written classification, audit, diagram, or decision rather than additional code.

**Notebook:** **Demo 5 is complete; do not rerun it.** Audit the saved
`inference_frontier` ledger.

1. Reproduce each policy's inference cost from its tokens and tool calls.
2. Verify expected total loss at $L_{wrong}=\$0.08$.
3. Recalculate the decision at $L_{wrong}=\$0.001$.
4. State how a one-second latency limit changes the feasible set.
5. Audit the tool route at 0%, 5%, and 10% router misses while retaining the
   2% invalid-argument rate.

**Research record:** planted accuracy; token prices; calls and tools; latency;
error loss; router and argument errors; selected policy; infeasible policies;
simulation boundary.

#### Your response

Replace the placeholders below with the requested output before opening the lecture debrief.

- **Result or artifact:** `[write here]`
- **Research-record fields:** `[write here]`
- **Bounded conclusion:** `[write here]`

## Where this leaves us

Context can specify a temporary task, retrieval can supply recoverable facts, adapters can change repeated behavior, and inference procedures can spend additional compute. None of those interventions is free, and none substitutes for checking the final claim.

Week 12 turns to decisions that alter a market environment: execution, simulation, and reinforcement learning.